In [ ]:
import os
import re
import pandas as pd


In [2]:
# -----------------------------
# Settings
# -----------------------------
in_folder = "Patent"
os.makedirs(in_folder, exist_ok=True)

files = [
    "chitosan.csv",
    "hydrogel.csv",
    "PCL.csv",
    "PEG.csv",
    "PLA.csv",
    "PLGA.csv",
    "poly_lactic_co_glycolic_acid.csv",
    "polycaprolactone.csv",
    "polyethylene_glycol.csv",
    "polylactic_acid.csv",
    "polymeric_micelle.csv",
]

out_clean = os.path.join(in_folder, "all_patents_cleaned.csv")
out_dropped = os.path.join(in_folder, "all_patents_dropped.csv")
out_summary = os.path.join(in_folder, "cleaning_summary.csv")

In [3]:
# Terms for filtering (lowercased matching)
DELIVERY_PATTERNS = [
    r"\bdrug[-\s]?delivery\b",
    r"\bdrug[-\s]?delivery\s+system(s)?\b",
    r"\bcontrolled[-\s]?release\b",
    r"\bsustained[-\s]?release\b",
    r"\btargeted[-\s]?delivery\b",
    r"\bdrug[-\s]?carrier(s)?\b",
    r"\btherapeutic[-\s]?delivery\b",
    r"\bnano[-\s]?particle(s)?\b",
    r"\bnano[-\s]?carrier(s)?\b",
    r"\bnano[-\s]?particulate(s)?\b",
    r"\bdrug\s+release\b",
    r"\brelease\s+profile(s)?\b",
]

POLYMER_TERMS = {
    "chitosan": ["chitosan"],
    "hydrogel": ["hydrogel", "hydrogels"],
    "PCL": ["pcl", "polycaprolactone"],
    "PEG": ["peg", "polyethylene glycol"],
    "PLA": ["pla", "polylactic acid", "poly(lactic acid)"],
    "PLGA": ["plga", "poly(lactic-co-glycolic acid)", "poly lactic co glycolic acid", "poly(lactide-co-glycolide)"],
    "poly_lactic_co_glycolic_acid": ["plga", "poly(lactic-co-glycolic acid)", "poly lactic co glycolic acid", "poly(lactide-co-glycolide)"],
    "polycaprolactone": ["polycaprolactone", "pcl"],
    "polyethylene_glycol": ["polyethylene glycol", "peg"],
    "polylactic_acid": ["polylactic acid", "pla", "poly(lactic acid)"],
    "polymeric_micelle": ["polymeric micelle", "polymeric micelles"],
}

# Standard polymer label for downstream merge with NIH
source_to_polymer = {
    "polycaprolactone": "PCL",
    "PCL": "PCL",
    "polylactic_acid": "PLA",
    "PLA": "PLA",
    "polyethylene_glycol": "PEG",
    "PEG": "PEG",
    "poly_lactic_co_glycolic_acid": "PLGA",
    "PLGA": "PLGA",
    "chitosan": "chitosan",
    "hydrogel": "hydrogel",
    "polymeric_micelle": "polymeric_micelle",
}

PRIMARY_COLS = [
    "#", "Jurisdiction", "Kind", "Display Key", "Lens ID",
    "Publication Date", "Publication Year", "Application Number", "Application Date",
    "Title", "Abstract", "Claims", "CPC Classifications", "IPCR Classifications",
    "Applicants/Assignees", "Inventors", "Legal Status"
]

In [5]:
 #-----------------------------
# Helpers (small, direct)
# -----------------------------
def to_str(x):
    return "" if pd.isna(x) else str(x)

def norm_text(x):
    s = to_str(x)
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

def combined_text(row):
    return " ".join([
        to_str(row.get("Title", "")),
        to_str(row.get("Abstract", "")),
        to_str(row.get("Claims", "")),
    ]).lower()

def has_delivery(text):
    return any(re.search(p, text) for p in DELIVERY_PATTERNS)

def keep_row(row, polymer_keywords):
    txt = combined_text(row)
    has_polymer = any(k in txt for k in polymer_keywords)
    return has_polymer and has_delivery(txt)


In [6]:
# -----------------------------
# Load all files
# -----------------------------
name_to_df = {}

for fname in files:
    path = os.path.join(in_folder, fname)
    if not os.path.exists(path):
        print("Missing file:", path)
        continue

    df = pd.read_csv(path)
    source = os.path.splitext(fname)[0]  # matches your original naming
    name_to_df[source] = df

if not name_to_df:
    raise ValueError("No patent files were loaded. Check folder and filenames.")

In [7]:
# -----------------------------
# Clean/filter each dataframe
# -----------------------------
kept_all = []
dropped_all = []
summary_rows = []

for source, df in name_to_df.items():
    df = df.copy()

    # keep primary cols first, keep everything else after
    keep_first = [c for c in PRIMARY_COLS if c in df.columns]
    rest = [c for c in df.columns if c not in keep_first]
    df = df[keep_first + rest]

    df["Source"] = source
    df["Polymer"] = source_to_polymer.get(source, source)

    # parse dates if present
    for c in ["Publication Date", "Application Date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")

    if "Publication Year" in df.columns:
        df["Publication Year"] = pd.to_numeric(df["Publication Year"], errors="coerce").astype("Int64")

    # normalize TAC fields
    for c in ["Title", "Abstract", "Claims"]:
        if c in df.columns:
            df[c] = df[c].map(norm_text)

    # drop rows where Title/Abstract/Claims all empty
    if all(c in df.columns for c in ["Title", "Abstract", "Claims"]):
        df = df[~((df["Title"] == "") & (df["Abstract"] == "") & (df["Claims"] == ""))]

    # dedupe within source
    if "Lens ID" in df.columns:
        df = df.drop_duplicates(subset=["Lens ID"])
    else:
        fallback_cols = [c for c in ["Title", "Publication Year"] if c in df.columns]
        if fallback_cols:
            df = df.drop_duplicates(subset=fallback_cols)

    polymer_keywords = [k.lower() for k in POLYMER_TERMS.get(source, [source])]
    mask_keep = df.apply(keep_row, axis=1, polymer_keywords=polymer_keywords)

    kept = df[mask_keep].copy()
    dropped = df[~mask_keep].copy()

    kept_all.append(kept)
    dropped_all.append(dropped)

    start_n = len(df)
    kept_n = len(kept)
    drop_n = len(dropped)

    summary_rows.append({
        "Source": source,
        "Start": start_n,
        "Kept": kept_n,
        "Dropped": drop_n,
        "Kept_%": round(100 * kept_n / start_n, 2) if start_n else 0.0
    })

    print(f"{source}: start {start_n} -> kept {kept_n}, dropped {drop_n}")


chitosan: start 3035 -> kept 1554, dropped 1481
hydrogel: start 4400 -> kept 2677, dropped 1723
PCL: start 591 -> kept 130, dropped 461
PEG: start 2664 -> kept 917, dropped 1747
PLA: start 936 -> kept 255, dropped 681
PLGA: start 1554 -> kept 527, dropped 1027
poly_lactic_co_glycolic_acid: start 794 -> kept 102, dropped 692
polycaprolactone: start 995 -> kept 206, dropped 789
polyethylene_glycol: start 4209 -> kept 1487, dropped 2722
polylactic_acid: start 1299 -> kept 612, dropped 687
polymeric_micelle: start 303 -> kept 61, dropped 242


In [8]:

# -----------------------------
# Merge and global dedupe
# -----------------------------
clean_df = pd.concat(kept_all, ignore_index=True)
dropped_df = pd.concat(dropped_all, ignore_index=True)

before = len(clean_df)

if "Lens ID" in clean_df.columns:
    clean_df = clean_df.drop_duplicates(subset=["Lens ID"])
else:
    fallback_cols = [c for c in ["Application Number", "Publication Year"] if c in clean_df.columns]
    if len(fallback_cols) == 2:
        clean_df = clean_df.drop_duplicates(subset=fallback_cols)
    else:
        fallback_cols = [c for c in ["Title", "Publication Year"] if c in clean_df.columns]
        if fallback_cols:
            clean_df = clean_df.drop_duplicates(subset=fallback_cols)

after = len(clean_df)
print(f"Deduplicated Patent dataset: {before} -> {after}")

print("Unique polymers:", sorted(clean_df["Polymer"].dropna().unique()))

# -----------------------------
# Save
# -----------------------------
clean_df.to_csv(out_clean, index=False)
dropped_df.to_csv(out_dropped, index=False)

summary = pd.DataFrame(summary_rows).sort_values(by="Source")
summary.to_csv(out_summary, index=False)

print("Saved:", out_clean)
print("Saved:", out_dropped)
print("Saved:", out_summary)


Deduplicated Patent dataset: 8528 -> 6684
Unique polymers: ['PCL', 'PEG', 'PLA', 'PLGA', 'chitosan', 'hydrogel', 'polymeric_micelle']
Saved: Patent\all_patents_cleaned.csv
Saved: Patent\all_patents_dropped.csv
Saved: Patent\cleaning_summary.csv
